In [10]:
import torch
import bitsandbytes as bnb
import transformers
import accelerate

print(torch.__version__)
print(bnb.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


2.1.2+cu121
0.43.3
4.39.3
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class ResumeRewriter:
    def __init__(self):
        model_id = "microsoft/Phi-3-mini-4k-instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0
        )

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            trust_remote_code=True,
            use_fast=True
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="cuda",
            trust_remote_code=True,
            attn_implementation="eager"
        ).eval()

    @torch.inference_mode()
    def rewrite_line(self, original_line, allowed_keywords):
        prompt = (
            f"Rewrite resume line.\n"
            f"Line: {original_line}\n"
            f"Keywords: {', '.join(allowed_keywords)}\n"
            "Rules: same meaning, one sentence.\n"
        )

        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        output = self.model.generate(
            **inputs,
            max_new_tokens=25,
            do_sample=False,
            use_cache=True
        )

        return self.tokenizer.decode(
            output[0][inputs.input_ids.shape[-1]:],
            skip_special_tokens=True
        )


c:\Users\Ayush\Desktop\model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rewriter = ResumeRewriter()

print(
    rewriter.rewrite_line(
        original_line="Built REST APIs using Node.js",
        allowed_keywords=["Docker", "AWS", "scalable"]
    )
)


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:12<00:00,  6.27s/it]
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48
You are not running the flash-attention implementation, expect numerical differences.



## Your task:

Revise the resume line to incorporate the keywords 'Docker' and '
